### モジュールのインポート

In [ ]:
import os
import glob
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import datetime
#from tqdm import tqdm
from tqdm.notebook import tqdm
import pickle
import random
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from PIL import Image
import skimage.transform
from collections import deque
from typing import Sequence, Dict, Tuple, Union

import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence
from torchvision import models
import torchvision.transforms as T
import torchvision.datasets as dataset
from torchvision.transforms import v2

from timm.scheduler import CosineLRScheduler
from transformers import  get_linear_schedule_with_warmup

#from transformers import AutoImageProcessor, AutoModel, AutoProcessor, CLIPVisionModel
from transformers import BertTokenizer, BertModel, CLIPVisionModel, BertForPreTraining

from evaluate import load
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from pycocoevalcap.cider.cider import Cider
from torchmetrics.multimodal import CLIPScore

import sys

import util
#from nltk import bleu_score
import ssl
from torch.amp import autocast, GradScaler

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend

### 位置エンコーディングの実装

In [ ]:
class PositionalEmbedding(nn.Module):
    '''
    位置埋め込み （Positional embedding）
    dim_embedding: 埋込み次元
    max_len      : 入力の最大系列長
    '''
    def __init__(self, dim_embedding: int, max_len: int=2048):
        super().__init__()

        self.pos_emb = nn.Embedding(max_len, dim_embedding)

    '''
    位置エンコーディングの順伝播
    x: 位置エンコーディングを埋め込む対象のテンソル,
       [バッチサイズ, 系列長, 埋め込み次元]
    '''
    def forward(self, x: torch.Tensor):
        seq = x.shape[1]
        positions = torch.arange(start=0, end=seq, step=1, device=x.device).to(torch.long)
        positions = self.pos_emb(positions)[:seq,:]
        
        return positions

### crf層定義

In [ ]:
def logsumexp(x, dim=1):
    return torch.logsumexp(x.float(), dim=dim).type_as(x)

class DynamicCRF(nn.Module):
    
    def __init__(self, num_embedding, low_rank=32, beam_size=64, crf_coef=1.0, temp = 1.0, top_p = 0.9, top_k = 50,
                 num_samples = 16, cand = 4, ref_t = False ):
        super().__init__()

        self.E1 = nn.Embedding(num_embedding, low_rank)
        self.E2 = nn.Embedding(num_embedding, low_rank)

        self.vocab_size = num_embedding
        self.rank = low_rank
        self.beam = beam_size
        self.crf_coef = crf_coef
        self.temp = temp
        self.top_p = top_p
        self.top_k = top_k
        self.num_samples = num_samples
        self.cand = cand
        self.ref_t = ref_t

    def _compute_score(self, emissions, targets, masks=None):
        batch_size, seq_len = targets.size()
        emission_scores = emissions.gather(2, targets[:, :, None])[:, :, 0]  # B x T
        transition_scores = (self.E1(targets[:, :-1]) * self.E2(targets[:, 1:])).sum(2)
        
        scores = emission_scores
        scores[:, 1:] += transition_scores
        
        #if masks is not None:
        #    scores = scores * masks.type_as(scores)

        return scores.sum(-1)
    
    def _compute_normalizer(self, emissions, targets=None, masks=None, beam=None):

        eps = 1e-8
        
        beam = beam if beam is not None else self.beam
        batch_size, seq_len = emissions.size()[:2]

        if targets is not None:
            #_emissions = emissions.scatter(2, targets[:, :, None], np.float('inf'))
            _emissions = emissions.scatter(2, targets[:, :, None], float('inf'))
            beam_targets = _emissions.topk(beam, 2)[1]
            beam_emission_scores = emissions.gather(2, beam_targets)
        else:
            beam_emission_scores, beam_targets = emissions.topk(beam, 2)
        beam_transition_score1 = self.E1(beam_targets[:, :-1])  # B x (T-1) x K x D; position i - 1, previous step.
        beam_transition_score2 = self.E2(beam_targets[:, 1:])   # B x (T-1) x K x D; position i, current step.
        beam_transition_matrix = torch.bmm(
            beam_transition_score1.view(-1, self.beam, self.rank),
            beam_transition_score2.view(-1, self.beam, self.rank).transpose(1, 2))
        beam_transition_matrix = beam_transition_matrix.view( batch_size, -1, self.beam, self.beam)

        # compute the normalizer in the log-space
        score = beam_emission_scores[:, 0]  # B x K
        for i in range(1, seq_len):
            next_score = score[:, :, None] + beam_transition_matrix[:, i-1]
            next_score = logsumexp(next_score, dim=1) + beam_emission_scores[:, i]
            tmp = logsumexp( next_score, dim = 1 )
            if masks is not None:
                score = torch.where(masks[:, i:i+1], next_score, score)
            else:
                score = next_score

        normalizer = logsumexp(score, dim=1)
        return normalizer

    def _compute_grpo_samples( self, beam_emission_scores, beam_transition_matrix, beam_targets, sampled_beam_idx=None, \
                          targets=None, masks=None, beam=None):

        eps = 1e-8
        device = beam_emission_scores.device

        permit_repeat = [ pad_token_id, eos_token_id, cls_token_id, sep_token_id, a_token_id, an_token_id, the_token_id, period_token_id, \
                         comma_token_id, and_token_id, in_token_id, we_token_id, i_token_id, he_token_id, she_token_id, \
                         it_token_id, they_token_id, dbl_token_id, sgl_token_id ]
        
        beam = beam if beam is not None else self.beam
        batch_size, seq_len, beam= beam_emission_scores.size()

        # フィルタリング用のパラメータ設定 (config等から取得できるよう適宜調整してください)
        #top_k = 50  # 上位k個に絞る (0なら無効)
        #top_p = 0.9 # 累積確率pまでに絞る (1.0なら無効)

        traj_tokens = []
        step_probs = []
      
        score = beam_emission_scores[:,0][:,None,:].expand( -1, self.num_samples, -1)
        #print( "1 score:", score )
        B, N, C, W = score.unsqueeze(-1).expand(-1,-1,-1,beam).shape
        #flat_score = score.unsqueeze(-1).expand(-1,-1,-1,beam).permute(0, 1, 3, 2).reshape(-1, C)
        #logits = flat_score / self.temp # B*N*W*cand, C
        #probs = F.softmax(logits, dim=-1)  # B*N*W*cand,C この softmax は、C についての softmax

        #torch.manual_seed(42)
        #_index_flat = torch.multinomial(probs, num_samples=self.cand, replacement=False)
        #_, _index_flat = torch.topk( probs, self,cand, -1)

        #score2 = torch.gather( probs, -1, _index_flat).view( B, N, W, self.cand)
        #score2 = torch.gather( flat_score, -1, _index_flat).view( B, N, W, self.cand)
        #score2 = torch.gather( flat_score / self.temp, -1, _index_flat).view( B, N, W, self.cand)
        #log_probs_t0 = torch.log_softmax( score / self.temp, dim=-1)
        logits_t0 = score / self.temp
        #score2 = score[:,:,:,None].expand(-1,-1,-1, self.cand)
        
        for i in range(1, seq_len):
            #_score = score.unsqueeze(-1) + beam_transition_matrix[:, i-1,None,:,:].expand( -1, N,-1,-1) #B,N,C,W

            _score_matrix = score.unsqueeze(-1) + beam_transition_matrix[:,i-1,None,:,:,].expand( -1, N, -1, -1 )
            _score_matrix = _score_matrix + beam_emission_scores[:,i][:,None,None,:].expand(-1,N,C,-1)

            # 【重要】強化学習用の遷移対数確率（次のBeam候補 dim=-1 に対する確率分布）
            # 形状: (bsz, beam, beam)
            #action_log_prob = torch.log_softmax(_score_matrix / self.temp, dim=-1)
            ##action_log_prob = _score_matrix
            ##action_log_prob = _score_matrix / self.temp
            #step_log_probs.append(action_log_prob)
            step_probs.append( _score_matrix / self.temp )
            
            _score, _index = _score_matrix.max( dim = 1 )
            
            B, N, C, W = _score_matrix.shape
            flat_score = _score_matrix.permute(0, 1, 3, 2 ).reshape(-1, C)

            # --- Top-K / Top-P Filtering 開始 (修正版) ---
            logits = flat_score / self.temp #B*W*N,C

            # 1. まず Top-K で上位K個に絞る (0なら無効)
            if self.top_k > 0:
                top_k_val = min(self.top_k, logits.size(-1)) # top_kが語彙数より大きくならないように調整
                top_k_logits, top_k_indices = torch.topk(logits, top_k_val, dim=-1)

                # Top-K用のマスクを作成
                min_values = top_k_logits[:, -1].unsqueeze(-1)
                logits = torch.where(logits < min_values, torch.full_like(logits, float('-inf')), logits)

                # 以降の処理（Top-P）のために top_k_logits, top_k_indices を更新
                # （注: top_pと組み合わせる場合、ここでのlogitsの更新より、
                #  後続のtop_k_indicesを使ったmaskの方がロジックが整合しやすい）

            # Top-Kの変数を再定義（top_pの処理で使うため）
            # top_k > 0 の場合、top_kで絞った後の値を使う。0の場合は全範囲。
            top_k_logits, top_k_indices = logits, torch.arange(logits.size(-1), device=logits.device).expand(logits.size(0), -1)
            # ↑ このアプローチはメモリを食うため、元の実装の通りtop_k_indicesでmaskする方が綺麗です。
            # 以下、元のロジックを活かした修正版です。
            # 2. Top-P (Nucleus) filtering
            if self.top_p < 1.0:
                # Top-K/Allで絞ったテンソルでソート
                sorted_logits, sorted_indices = torch.sort(top_k_logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

                # 除去対象のマスクを作成
                sorted_indices_to_remove = cumulative_probs > self.top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0

                # top_k_logitsと同じ形状のマスクを作成
                indices_to_remove_k = torch.zeros_like(top_k_logits, dtype=torch.bool).scatter_(
                    dim=-1, index=sorted_indices, src=sorted_indices_to_remove
                )
                top_k_logits[indices_to_remove_k] = -float('Inf')

            # 3. 全語彙の logits を一旦すべて -inf にして、生き残った top_k_logits だけを戻す
            new_logits = torch.full_like(logits, float('-inf'))
            new_logits.scatter_(dim=-1, index=top_k_indices, src=top_k_logits)
            logits = new_logits

            if torch.all(logits == float('inf')):
                logits = torch.ones_like(logits)  # テンソル全体を1にする場合
            
            probs = F.softmax(logits, dim=-1) # B*W*N,C この softmax は、C についての softmax

            #torch.manual_seed(42)
            _index_flat = torch.multinomial(probs, num_samples=self.cand, replacement=False) #B*N*W,cand
            #_index_flat = torch.topk( probs, self.cand, dim = -1 )
            _index_flat1 = _index_flat[:,:1]

            #_score_flat = torch.gather( probs, -1, _index_flat1)
            #_score_flat2 = torch.gather( probs, -1, _index_flat)
            #_score_flat = torch.gather( flat_score / self.temp, -1, _index_flat1)
            #_score_flat2 = torch.gather( flat_score / self.temp, -1, _index_flat)
            _score_flat = torch.gather( flat_score, -1, _index_flat1)
            #_score_flat2 = torch.gather( flat_score, -1, _index_flat)
            _index = _index_flat.view(B, N, W, self.cand) # B, N, W, cand
            _score = _score_flat.view(B, N, W) # B, N, W
            #_score2 = _score_flat2.view( B, N, W, self.cand )
            
            #_score = _score + beam_emission_scores[:, i][:,None,:].expand(-1,N,-1)
            score = _score
            #_score2 = _score2 + beam_emission_scores[:,i][:,None,:,None].expand( -1, N, -1, self.cand )
            #score2 = _score2
            
            traj_tokens.append( _index ) # S, B, N, W, cand

        ## 2. サンプリングされた最後のインデックスを取得 (B, N, 1)
        #print( "_score:", _score )
        B, N, C = score.shape
        flat_score = score.reshape(-1, C)

        probs = F.softmax(flat_score / self.temp, dim=-1)
        #torch.manual_seed(42)
        #print( "probs:", probs )
        _index_flat = torch.multinomial(probs, num_samples=self.cand, replacement = False ) #B*N,cand
        #_index_flat = torch.topk( probs, self.cand, -1 )

        #_score_flat = torch.gather(probs, -1, _index_flat)
        #_score_flat = torch.gather(flat_score, -1, _index_flat)
        _index = _index_flat.view(B, N, self.cand)
        #_score = _score_flat.view(B, N, self.cand)
        #_score = _score.unsqueeze(2).expand(-1,-1,W,-1)
        current_sampled_index = _index #(B, N, cand )

        # beam から vocab_size に戻す。
        N_beam_targets = beam_targets[:,-1].unsqueeze( 1 ).expand( -1, N, -1 ) # B, N, W
        current_sampled_index = torch.gather( N_beam_targets, -1, current_sampled_index ) #B,N,cand

        finalized_tokens = torch.full( (seq_len,B,N), self.vocab_size , dtype=torch.long, device=device)

        ## 3. 最初の要素として追加
        finalized_tokens[0] = current_sampled_index[:,:,0]
        #traj_scores = torch.stack( traj_scores, dim = 0 )
        traj_tokens = torch.stack( traj_tokens, dim = 0 )

        # beam から vocab_size に戻す。
        N_beam_targets = beam_targets.unsqueeze( 2 ).unsqueeze(4).expand( -1, -1, N, -1, self.cand ) # B, seq_len, N, W
        N_beam_targets = N_beam_targets.permute( 1, 0, 2, 4, 3 ) #S,B,N,cand,W
        N_beam_targets1 = N_beam_targets[:-1]
        N_beam_targets2 = N_beam_targets[1:]
        ##traj_tokens S,B,N,W,cand, traj_scores S,B,N,W
        traj_tokens = torch.gather( N_beam_targets1, -1, traj_tokens.transpose(3,4))
        traj_tokens3 = torch.full( ( seq_len - 1, B, N, self.cand, self.vocab_size ), self.vocab_size, dtype=torch.long, device = beam_targets.device )
        traj_tokens3 = torch.scatter( traj_tokens3, -1, index = N_beam_targets2, src = traj_tokens)

        # バックトレーシング
        if sampled_beam_idx is not None:
            flag = False
            beam_logits = []
        else:
            flag = True
        #for i3, (idx_step, prob_step ) in enumerate( zip(torch.flip(traj_tokens3, dims=(0,)),torch.flip(traj_scores, dims=(0,)) ) ):
        for i3, (idx_step, step_prob ) in enumerate( zip(torch.flip(traj_tokens3, dims=(0,)), reversed( step_probs ))):
            if sampled_beam_idx is not None:
                i4 = seq_len - i3 - 1
                previous_beam_index = sampled_beam_idx[:,i4-1]
                prob_from_prev = step_prob.gather(2, previous_beam_index[:, :, None, None].expand(-1, -1, 1,  beam))
                beam_logits.append( prob_from_prev.squeeze(2) ) # (S), B, W
            
            i2= i3+1
            previous_pointer = finalized_tokens[i3]
            stop_flag = torch.zeros( (B,N), dtype=torch.int,device=device)
            for i in range( self.cand ):
                #cand_probs = prob_step[:,:,:,i]
                idx = idx_step[:,:,i].new_empty((idx_step[:,:,i].size(0), idx_step[:,:,i].size(1), idx_step[:,:,i].size(2)+1))
                idx[:,:,:-1] = idx_step[:,:,i]
                idx[:,:,idx.size(2)-1] = self.vocab_size
                cand_tokens = torch.gather( idx,-1, previous_pointer.unsqueeze(-1) ).squeeze(-1)# B,N
                finalized_tokens2 = finalized_tokens.clone()
                finalized_tokens2[ i2: ] = -100
                repeat_mask = ( finalized_tokens2 == cand_tokens  )  # S, B , N
                not_permit_mask = (~torch.isin( finalized_tokens2, torch.tensor( permit_repeat, device=device))).to(torch.int ) 
                repeat_sum = ( (repeat_mask).to(torch.int) * not_permit_mask ).sum(dim =0 )# B,N
                if i == self.cand -1:#最終の時は、
                    chg_flag = ~(stop_flag.to(torch.bool)) #最終の前までに stop が1になれば変えない。stop が0だったら変える。
                    if i == 0:
                        chg_flag = torch.ones( (B,N), dtype=torch.bool,device=device)
                else:
                    tmp_flag =  stop_flag + repeat_sum  # stopとrepeat両方が0の時 0
                    chg_flag = ( tmp_flag == 0 )# stop と　repeat両方が0の時　chgは　true
                    stop_flag[ stop_flag == 1 ] = 1
                    stop_flag[ chg_flag ] = 1
                finalized_tokens[i2] = torch.where(chg_flag,cand_tokens,finalized_tokens[i2])#(S),B,N True だったら変更、 False だったらそのまま。
                chg_flag2 = chg_flag.unsqueeze(2).expand(-1,-1,W)

        finalized_tokens = torch.flip( finalized_tokens, dims= (0, ) )
        finalized_tokens = finalized_tokens.transpose( 0, 1 )#B,S,N
        
        # vocab_size のfinalized_tokens から beam の sampled_beam_idx を作る
        N_beam_targets = beam_targets.unsqueeze( 2 ).expand( -1, -1, N, -1 ) # B, seq_len, N, W
        mask = N_beam_targets == finalized_tokens.unsqueeze(-1) #B,S,N,W
        # 2. ビーム次元 (-1) で一致しているインデックスを取得
        sampled_beam_idx = torch.argmax(mask.to(torch.int32), dim=-1)

        if flag == True:
            beam_logits = []
            for i3, step_prob in enumerate( reversed( step_probs )):
                i4 = seq_len - i3 - 1
                previous_beam_index = sampled_beam_idx[:,i4-1]
                prob_from_prev = step_prob.gather(2, previous_beam_index[:, :, None, None].expand(-1, -1, 1,  beam))
                beam_logits.append( prob_from_prev.squeeze(2) ) # (S), B, W
        
        #print( "log_beam_logits:", log_beam_logits )
        beam_logits.append( logits_t0 )
        beam_logits = torch.stack( beam_logits, dim = 0 )
        beam_logits = torch.flip( beam_logits, dims = (0, ) )
        beam_logits = beam_logits.permute( 1, 0, 3, 2 ) # B,S,W,N
        beam_probs = F.softmax( beam_logits, dim = 2 )

        log_beam_probs = torch.log( beam_probs + eps )
    
        
        return beam_probs, log_beam_probs, sampled_beam_idx, finalized_tokens

    def _compute_ref_probs( self, beam_emission_scores, beam_transition_matrix, beam_targets, sampled_beam_idx = None,  
                            use_crf_beam_logits = False, masks = None, beam = None ):
        
        eps = 1e-8
        device = beam_emission_scores.device

        permit_repeat = [ pad_token_id, eos_token_id, cls_token_id, sep_token_id, a_token_id, an_token_id, the_token_id, period_token_id, \
                         comma_token_id, and_token_id, in_token_id, we_token_id, i_token_id, he_token_id, she_token_id, \
                         it_token_id, they_token_id, dbl_token_id, sgl_token_id ]

        beam = beam if beam is not None else self.beam
        batch_size, seq_len, beam= beam_emission_scores.size()

        #traj_scores = []
        traj_tokens = []
        #step_log_probs = []
        step_probs = []

        score = beam_emission_scores[:,0][:,None,:].expand( -1, self.num_samples, -1)
        B, N, C, W = score.unsqueeze(-1).expand(-1,-1,-1,beam).shape
        #flat_score = score.unsqueeze(-1).expand(-1,-1,-1,beam).permute(0, 1, 3, 2).reshape(-1, C)
        #logits = flat_score / self.temp # B*N*W*cand, C
        #probs = F.softmax(logits, dim=-1)  # B*N*W*cand,C この softmax は、C についての softmax

        #torch.manual_seed(42)
        #_index_flat = torch.multinomial(probs, num_samples=self.cand, replacement=False)
        #_, _index_flat = torch.topk( probs, self,cand, -1)

        #score2 = torch.gather( probs, -1, _index_flat).view( B, N, W, self.cand)
        #score2 = torch.gather( flat_score / self.temp, -1, _index_flat).view( B, N, W, self.cand)
        #score2 = torch.gather( flat_score, -1, _index_flat).view( B, N, W, self.cand)
        #log_probs_t0 = torch.log_softmax( score / self.temp, dim=-1)
        logits_t0 = score / self.temp
        #score2 = score[:,:,:,None].expand(-1,-1,-1, self.cand)
        
        for i in range(1, seq_len):
            #traj_scores.append( score2 ) # S,B,N,W
            #_score = score.unsqueeze(-1) + beam_transition_matrix[:, i-1,None,:,:].expand( -1, N,-1,-1) #B,N,C,W

            _score_matrix = score.unsqueeze(-1) + beam_transition_matrix[:,i-1,None,:,:,].expand( -1, N, -1, -1 )
            _score_matrix = _score_matrix + beam_emission_scores[:,i][:,None,None,:].expand(-1,N,C,-1)

            # 【重要】強化学習用の遷移対数確率（次のBeam候補 dim=-1 に対する確率分布）
            # 形状: (bsz, beam, beam)
            #action_log_prob = torch.log_softmax(_score_matrix / self.temp, dim=-1)
            #action_log_prob = _score_matrix
            #action_log_prob = _score_matrix / self.temp
            #step_log_probs.append(action_log_prob)
            step_probs.append( _score_matrix / self.temp )
            
            _score, _index = _score_matrix.max( dim = 1 )
            
            B, N, C, W = _score_matrix.shape
            flat_score = _score_matrix.permute(0, 1, 3, 2 ).reshape(-1, C)

            # --- Top-K / Top-P Filtering 開始 (修正版) ---
            logits = flat_score / self.temp #B*W*N,C

            probs = F.softmax(logits, dim=-1) # B*W*N,C この softmax は、C についての softmax

            #torch.manual_seed(42)
            _index_flat = torch.multinomial(probs, num_samples=self.cand, replacement=False) #B*N*W,cand
            #_index_flat = torch.topk( probs, self.cand, dim = -1 )
            _index_flat1 = _index_flat[:,:1]

            _score_flat = torch.gather( probs, -1, _index_flat1)
            #_score_flat2 = torch.gather( probs, -1, _index_flat)
            #_score_flat = torch.gather( flat_score / self.temp, -1, _index_flat1)
            #_score_flat2 = torch.gather( flat_score / self.temp, -1, _index_flat)
            _score_flat = torch.gather( flat_score, -1, _index_flat1)
            #_score_flat2 = torch.gather( flat_score, -1, _index_flat)
            _index = _index_flat.view(B, N, W, self.cand) # B, N, W, cand
            _score = _score_flat.view(B, N, W) # B, N, W
            #_score2 = _score_flat2.view( B, N, W, self.cand )
            
            #_score = _score + beam_emission_scores[:, i][:,None,:].expand(-1,N,-1)
            score = _score
            #_score2 = _score2 + beam_emission_scores[:,i][:,None,:,None].expand( -1, N, -1, self.cand )
            #score2 = _score2
            
            traj_tokens.append( _index ) # S, B, N, W, cand
            
        ## 2. サンプリングされた最後のインデックスを取得 (B, N, 1)
        B, N, C = score.shape
        flat_score = score.reshape(-1, C)

        probs = F.softmax(flat_score / self.temp, dim=-1)
        #torch.manual_seed(42)
        _index_flat = torch.multinomial(probs, num_samples=self.cand, replacement = False ) #B*N,cand
        #_index_flat = torch.topk( probs, self.cand, -1 )

        #_score_flat = torch.gather(probs, -1, _index_flat)
        #_score_flat = torch.gather(flat_score, -1, _index_flat)
        _index = _index_flat.view(B, N, self.cand)
        #_score = _score_flat.view(B, N, self.cand)
        #_score = _score.unsqueeze(2).expand(-1,-1,W,-1)
        current_sampled_index = _index #(B, N, cand )

        # beam から vocab_size に戻す。
        N_beam_targets = beam_targets[:,-1].unsqueeze( 1 ).expand( -1, N, -1 ) # B, N, W
        current_sampled_index = torch.gather( N_beam_targets, -1, current_sampled_index ) #B,N,cand

        finalized_tokens = torch.full( (seq_len,B,N), self.vocab_size , dtype=torch.long, device=device)

        ## 3. 最初の要素として追加
        finalized_tokens[0] = current_sampled_index[:,:,0]
        #traj_scores = torch.stack( traj_scores, dim = 0 )

        traj_tokens = torch.stack( traj_tokens, dim = 0 )

        # beam から vocab_size に戻す。
        N_beam_targets = beam_targets.unsqueeze( 2 ).unsqueeze(4).expand( -1, -1, N, -1, self.cand ) # B, seq_len, N, W
        N_beam_targets = N_beam_targets.permute( 1, 0, 2, 4, 3 ) #S,B,N,cand,W
        N_beam_targets1 = N_beam_targets[:-1]
        N_beam_targets2 = N_beam_targets[1:]
        ##traj_tokens S,B,N,W,cand, traj_scores S,B,N,W
        traj_tokens = torch.gather( N_beam_targets1, -1, traj_tokens.transpose(3,4))
        traj_tokens3 = torch.full( ( seq_len - 1, B, N, self.cand, self.vocab_size ), self.vocab_size, dtype=torch.long, device = beam_targets.device )
        traj_tokens3 = torch.scatter( traj_tokens3, -1, index = N_beam_targets2, src = traj_tokens)

        # バックトレーシング
        beam_logits = []
        #log_beam_probs = []
        for i3, (idx_step,step_prob ) in enumerate( zip(torch.flip(traj_tokens3, dims=(0,)),reversed( step_probs ))):
            i4 = seq_len - i3 - 1
            previous_beam_index = sampled_beam_idx[:,i4-1]
            prob_from_prev = step_prob.gather(2, previous_beam_index[:, :, None, None].expand(-1, -1, 1,  beam))
            beam_logits.append( prob_from_prev.squeeze(2) ) # (S), B, W       
            
            i2= i3+1
            previous_pointer = finalized_tokens[i3]
            stop_flag = torch.zeros( (B,N), dtype=torch.int,device=device)
            for i in range( self.cand ):
                #cand_probs = prob_step[:,:,:,i]
                idx = idx_step[:,:,i].new_empty((idx_step[:,:,i].size(0), idx_step[:,:,i].size(1), idx_step[:,:,i].size(2)+1))
                idx[:,:,:-1] = idx_step[:,:,i]
                idx[:,:,idx.size(2)-1] = self.vocab_size
                cand_tokens = torch.gather( idx,-1, previous_pointer.unsqueeze(-1) ).squeeze(-1)# B,N
                finalized_tokens2 = finalized_tokens.clone()
                finalized_tokens2[ i2: ] = -100
                repeat_mask = ( finalized_tokens2 == cand_tokens  )  # S, B , N
                not_permit_mask = (~torch.isin( finalized_tokens2, torch.tensor( permit_repeat, device=device))).to(torch.int ) 
                repeat_sum = ( (repeat_mask).to(torch.int) * not_permit_mask ).sum(dim =0 )# B,N
                if i == self.cand -1:#最終の時は、
                    chg_flag = ~(stop_flag.to(torch.bool)) #最終の前までに stop が1になれば変えない。stop が0だったら変える。
                    if i == 0:
                        chg_flag = torch.ones( (B,N), dtype=torch.bool,device=device)
                else:
                    tmp_flag =  stop_flag + repeat_sum  # stopとrepeat両方が0の時 0
                    chg_flag = ( tmp_flag == 0 )# stop と　repeat両方が0の時　chgは　true
                    stop_flag[ stop_flag == 1 ] = 1
                    stop_flag[ chg_flag ] = 1
                finalized_tokens[i2] = torch.where(chg_flag,cand_tokens,finalized_tokens[i2])#(S),B,N True だったら変更、 False だったらそのまま。
                chg_flag2 = chg_flag.unsqueeze(2).expand(-1,-1,W)

        beam_logits.append( logits_t0 )
        beam_logits = torch.stack( beam_logits, dim = 0 )
        beam_logits = torch.flip( beam_logits, dims = (0, ) )
        beam_logits = beam_logits.permute( 1, 0, 3, 2 ) # B,S,W,N
        ref_log_beam_probs = F.log_softmax( beam_logits, dim = 2 )
        
        finalized_tokens = torch.flip( finalized_tokens, dims= (0, ) )
        finalized_tokens = finalized_tokens.transpose( 0, 1 )#B,S,N
       
        return ref_log_beam_probs, finalized_tokens
    
    
    def forward(self, emissions, targets, sampled_beam_idx = None, top_indices = None, grpo_mode = True, crf_mode = False, 
                use_crf_beam_logits = False, masks=None, beam=None):

        beam = beam if beam is not None else self.beam
        batch_size, seq_len = emissions.size()[:2]
        B = batch_size
        device = emissions.device
        permit_repeat = [ pad_token_id, eos_token_id, cls_token_id, sep_token_id, a_token_id, an_token_id, the_token_id, period_token_id, \
                         comma_token_id, and_token_id, in_token_id, we_token_id, i_token_id, he_token_id, she_token_id, \
                         it_token_id, they_token_id, dbl_token_id, sgl_token_id ]
        
        if top_indices == None:
            beam_emission_scores, beam_targets = torch.topk( emissions, beam, -1)
        else:
            beam_emission_scores = torch.gather( emissions, -1, top_indices )
            beam_targets = top_indices
        
        beam_transition_score1 = self.E1(beam_targets[:, :-1])  # B x (T-1) x K x D
        beam_transition_score2 = self.E2(beam_targets[:, 1:])   # B x (T-1) x K x D
        beam_transition_matrix = torch.bmm(
            beam_transition_score1.view(-1, self.beam, self.rank),
            beam_transition_score2.view(-1, self.beam, self.rank).transpose(1, 2))
        beam_transition_matrix = beam_transition_matrix.view(batch_size, -1, beam, beam) # bsz, seq_len, beam, beam

        if not self.ref_t:

            traj_tokens = []
            step_probs = []

            # compute the normalizer in the log-space
            score = beam_emission_scores[:, 0]  # B x K
            #dummy = torch.arange(beam, device=score.device).expand(*score.size()).contiguous()

            logits_t0 = score  / self.temp            
            
            for i in range(1, seq_len):
                _score_matrix = score.unsqueeze(-1) + beam_transition_matrix[:,i-1,:,:,].expand( -1, -1, -1 )
                _score_matrix = _score_matrix + beam_emission_scores[:,i][:,None,:].expand(-1,beam,-1)

                step_probs.append( _score_matrix / self.temp )

                _score2, _index2 = torch.topk( _score_matrix, self.cand, dim = 1 ) 
                _score = _score2[:,0]
                _index = _index2[:,0]
                
                #if masks is not None:
                #    score = torch.where(masks[:, i: i+1], _score, score)
                #    index = torch.where(masks[:, i: i+1], _index, dummy)
                #else:
                score, index = _score, _index
                traj_tokens.append(_index2) # S, B, cand, W

            _, _indexes = torch.topk( score, self.cand, dim = 1 )
            current_sampled_index = _indexes #(B,cand )

            # beam から vocab_size に戻す。
            beam_targets1 = beam_targets[:,-1] # B, W
            current_sampled_index = torch.gather( beam_targets1, -1, current_sampled_index ) #B,cand
        
            finalized_tokens = torch.full( (seq_len,B), self.vocab_size , dtype=torch.long, device=device)
    
            ## 3. 最初の要素として追加
            finalized_tokens[0] = current_sampled_index[:,0] # (S), B

            traj_tokens = torch.stack( traj_tokens, dim = 0 ) # S, B, cand, W
            
            # beam から vocab_size に戻す。
            cand_beam_targets = beam_targets.unsqueeze(2).expand( -1, -1, self.cand, -1 ) # B, seq_len, cand, W
            cand_beam_targets = cand_beam_targets.permute( 1, 0, 2, 3 ) #S,B,cand,W
            cand_beam_targets1 = cand_beam_targets[:-1]
            cand_beam_targets2 = cand_beam_targets[1:]
            traj_tokens = torch.gather( cand_beam_targets1, -1, traj_tokens)
            traj_tokens3 = torch.full( ( seq_len - 1, B, self.cand, self.vocab_size ), self.vocab_size, dtype=torch.long, device = beam_targets.device )
            traj_tokens3 = torch.scatter( traj_tokens3, -1, index = cand_beam_targets2, src = traj_tokens)

            for i3, idx_step in enumerate( torch.flip(traj_tokens3, dims=(0,))):
                i2= i3+1
                previous_pointer = finalized_tokens[i3]
                stop_flag = torch.zeros( (B), dtype=torch.int,device=device) # stop_flag が 0 の時 更新 OK, 1の時、これ以上更新しない。
                for i in range( self.cand ):
                    idx = idx_step[:,i].new_empty((idx_step[:,i].size(0), idx_step[:,i].size(1)+1 ))
                    idx[:,:-1] = idx_step[:,i]
                    idx[:,idx.size(1)-1] = self.vocab_size
                    cand_tokens = torch.gather( idx,-1, previous_pointer.unsqueeze(-1) ).squeeze(-1)# B,N　更新の候補を作成。
                    finalized_tokens2 = finalized_tokens.clone()
                    finalized_tokens2[ i2: ] = -100 # 現在の時刻より先は -100
                    repeat_mask = ( finalized_tokens2 == cand_tokens  )  # S, B 繰り返しの場所を特定する　mask 
                    not_permit_mask = (~torch.isin( finalized_tokens2, torch.tensor( permit_repeat, device=device))).to(torch.int ) 
                    repeat_sum = ( (repeat_mask).to(torch.int) * not_permit_mask ).sum(dim =0 )# B 繰り返しの場所に繰り返しが許可されたトークンの場所をかけて和をとることにより、B の形状の繰り返しがある時1 以上、ない時0 を 得る。
                    if i == self.cand -1:#最終の時は、
                        chg_flag = ~(stop_flag.to(torch.bool)) #最終の前までに stop が1になれば変えない。stop が0だったら変える。
                        if i == 0:
                            chg_flag = torch.ones( (B), dtype=torch.bool,device=device)
                    else:
                        tmp_flag =  stop_flag + repeat_sum  # stopとrepeat両方が0の時 0
                        chg_flag = ( tmp_flag == 0 )# stop と　repeat両方が0の時　chgは　true
                        stop_flag[ stop_flag == 1 ] = 1 # stop_flag が 1の場合は stop_flag =1
                        stop_flag[ chg_flag ] = 1 #chg_flag = True の場合は、更新されたのだから stop_flag = 1
                    finalized_tokens[i2] = torch.where(chg_flag,cand_tokens,finalized_tokens[i2])#(S),B True だったら変更、 False だったらそのまま。
                    chg_flag2 = chg_flag.unsqueeze(1).expand(-1,beam)
            
            finalized_tokens = torch.flip( finalized_tokens, dims= (0, ) )
            finalized_tokens = finalized_tokens.transpose( 0, 1 )#B,S,N

            if use_crf_beam_logits:
                # vocab_size のfinalized_tokens から beam の sampled_beam_idx を作る
                mask = beam_targets == finalized_tokens.unsqueeze(-1) #B,S,W
                # 2. ビーム次元 (-1) で一致しているインデックスを取得
                sampled_beam_idx2 = torch.argmax(mask.to(torch.int32), dim=-1)
        
                beam_logits = []
                #print( "define log_beam_probs None:")
                for i3, probs_step in enumerate( reversed( step_probs )):
                    i4 = seq_len - i3 - 1
                    #current_beam_index = sampled_beam_idx2[:,i4]
                    previous_beam_index = sampled_beam_idx2[:,i4-1]
                    prob_from_prev = probs_step.gather(2, previous_beam_index[:, None, None].expand(-1, 1,  beam))
                    beam_logits.append( prob_from_prev.squeeze(1) )
        
                beam_logits.append( logits_t0 )
                beam_logits = torch.stack( beam_logits, dim = 0 )
                beam_logits = torch.flip( beam_logits, dims = (0, ) )
                beam_logits = beam_logits.permute( 1, 0, 2 )
                crf_beam_logits = beam_logits            
            else:
                crf_beam_logits = torch.tensor( [0] )
    
        if not self.ref_t:
            if grpo_mode:
                if not use_crf_beam_logits:
                    numerator = self._compute_score(emissions, targets)
                    denominator = self._compute_normalizer(emissions, targets)
                    crf_loss = - ( numerator - denominator ).mean() / seq_len
                else:
                    crf_loss = torch.tensor( [0] )
                beam_probs, log_beam_probs, sampled_beam_idx, b_finalized_tokens,  = \
                    self._compute_grpo_samples( beam_emission_scores, beam_transition_matrix, beam_targets, sampled_beam_idx = sampled_beam_idx )
                return  finalized_tokens, beam_probs, log_beam_probs, crf_loss, crf_beam_logits, beam_targets, sampled_beam_idx, b_finalized_tokens
            else:
                if crf_mode:
                    if not use_crf_beam_logits:
                        numerator = self._compute_score(emissions, targets)
                        denominator = self._compute_normalizer(emissions, targets)
                        crf_loss = - ( numerator - denominator ).mean() / seq_len
                    else:
                        crf_loss = torch.tensor( [0] )
                    return finalized_tokens, crf_loss, crf_beam_logits, beam_targets
                else:
                    return finalized_tokens
        else:
            ref_log_beam_probs, finalized_tokens = self._compute_ref_probs( beam_emission_scores, \
                                    beam_transition_matrix, beam_targets, sampled_beam_idx )
            return ref_log_beam_probs, finalized_tokens, beam_targets

    '''original
    def _viterbi_decode(self, emissions, masks=None, beam=None):
        beam = beam if beam is not None else self.beam
        batch_size, seq_len = emissions.size()[:2]
        beam_emission_scores, beam_targets = emissions.topk(beam, 2)
        beam_transition_score1 = self.E1(beam_targets[:, :-1])  # B x (T-1) x K x D
        beam_transition_score2 = self.E2(beam_targets[:, 1:])   # B x (T-1) x K x D
        beam_transition_matrix = torch.bmm(
            beam_transition_score1.view(-1, beam, self.rank),
            beam_transition_score2.view(-1, beam, self.rank).transpose(1, 2))
        beam_transition_matrix = beam_transition_matrix.view(batch_size, -1, beam, beam) # bsz, seq_len, beam, beam

        traj_tokens, traj_scores = [], []
        finalized_tokens, finalized_scores = [], []

        # compute the normalizer in the log-space
        score = beam_emission_scores[:, 0]  # B x K
        #print( "score size:", score.size() )
        dummy = torch.arange(beam, device=score.device).expand(*score.size()).contiguous()

        for i in range(1, seq_len):
            traj_scores.append(score)
            _score = score[:, :, None] + beam_transition_matrix[:, i-1] # bsz, beam, beam
            _score, _index = _score.max(dim=1) # bsz, beam     bsz, beam 
            _score = _score + beam_emission_scores[:, i] # bsz, beam

            if masks is not None:
                score = torch.where(masks[:, i: i+1], _score, score)
                index = torch.where(masks[:, i: i+1], _index, dummy)
            else:
                score, index = _score, _index
            traj_tokens.append(index)

        # now running the back-tracing and find the best
        best_score, best_index = score.max(dim=1)
        finalized_tokens.append(best_index[:, None])
        finalized_scores.append(best_score[:, None])

        for idx, scs in zip(reversed(traj_tokens), reversed(traj_scores)):
            previous_index = finalized_tokens[-1]
            finalized_tokens.append(idx.gather(1, previous_index))
            finalized_scores.append(scs.gather(1, previous_index))

        finalized_tokens.reverse()
        finalized_tokens = torch.cat(finalized_tokens, 1)
        finalized_tokens = beam_targets.gather(2, finalized_tokens[:, :, None])[:, :, 0]

        finalized_scores.reverse()
        finalized_scores = torch.cat(finalized_scores, 1)
        finalized_scores[:, 1:] = finalized_scores[:, 1:] - finalized_scores[:, :-1]

        return finalized_scores, finalized_tokens
    '''
    def _viterbi_decode(self, emissions, masks=None, beam=None):
        beam = beam if beam is not None else self.beam
        batch_size, seq_len = emissions.size()[:2]
        beam_emission_scores, beam_targets = emissions.topk(beam, 2)
        beam_transition_score1 = self.E1(beam_targets[:, :-1])  # B x (T-1) x K x D
        beam_transition_score2 = self.E2(beam_targets[:, 1:])   # B x (T-1) x K x D
        beam_transition_matrix = torch.bmm(
            beam_transition_score1.view(-1, beam, self.rank),
            beam_transition_score2.view(-1, beam, self.rank).transpose(1, 2))
        beam_transition_matrix = beam_transition_matrix.view(batch_size, -1, beam, beam) # bsz, seq_len, beam, beam

        traj_tokens, traj_scores = [], []
        finalized_tokens, finalized_scores = [], []
        
        # 時刻ごとの遷移確率行列（log_softmax）を保存するリスト
        # 各要素の形状: (bsz, beam, beam) -> [batch, from_beam_idx, to_beam_idx]
        step_log_probs = []

        # compute the normalizer in the log-space
        score = beam_emission_scores[:, 0]  # B x K
        dummy = torch.arange(beam, device=score.device).expand(*score.size()).contiguous()
        
        # 時刻 t=0 の初期選択確率（最高位候補の中でのlog_softmax）
        # 形状: (bsz, beam)
        log_probs_t0 = torch.log_softmax(score / self.temp, dim=-1)

        for i in range(1, seq_len):
            traj_scores.append(score)
            
            # _score_matrix 形状: (bsz, beam, beam) -> [batch, from_beam, to_beam]
            _score_matrix = score[:, :, None] + beam_transition_matrix[:, i-1] 
            _score_matrix = _score_matrix + beam_emission_scores[:, i][:, None, :]
            
            # 【重要】強化学習用の遷移対数確率（次のBeam候補 dim=-1 に対する確率分布）
            # 形状: (bsz, beam, beam)
            action_log_prob = torch.log_softmax(_score_matrix / self.temp, dim=-1)
            #action_log_prob = _score_matrix
            #action_log_prob = _score_matrxi / self.temp
            step_log_probs.append(action_log_prob)

            # 通常のビタビ処理
            _score, _index = _score_matrix.max(dim=1) # bsz, beam

            if masks is not None:
                score = torch.where(masks[:, i: i+1], _score, score)
                index = torch.where(masks[:, i: i+1], _index, dummy)
            else:
                score, index = _score, _index
            traj_tokens.append(index)

        traj_scores.append(score)

        # --- バックトラック（逆順のトレース） ---
        best_score, best_index = score.max(dim=1)
        finalized_tokens.append(best_index[:, None])
        finalized_scores.append(best_score[:, None])
        
        # 強化学習用の確率を後ろから格納していくリスト
        sampled_log_probs_reversed = []

        # zipの中身: i が大きい（未来）順にループが回る
        # idx: 時刻 i で選択された「前の時刻のBeamインデックス」 (bsz, beam)
        # scs: 時刻 i-1 での累積スコア (bsz, beam)
        # step_probs: 時刻 i-1 から i への遷移対数確率 (bsz, beam, beam)
        for idx, scs, step_probs in zip(reversed(traj_tokens), reversed(traj_scores), reversed(step_log_probs)):
            # 現在（時刻 i）確定している最適パスの「現在のBeamインデックス」 (bsz, 1)
            current_beam_index = finalized_tokens[-1]
            
            # 1. 1つ前の時刻の最適Beamインデックスを取得
            previous_beam_index = idx.gather(1, current_beam_index)
            finalized_tokens.append(previous_beam_index)
            finalized_scores.append(scs.gather(1, current_beam_index))
            
            # 2. 【重要】強化学習用確率の抽出
            # step_probs は (bsz, from_beam, to_beam)
            # a) まず from_beam (dim=1) を previous_beam_index で指定して抽出 -> (bsz, 1, beam)
            prob_from_prev = step_probs.gather(1, previous_beam_index[:, :, None].expand(-1, -1, beam))
            # b) 次に to_beam (dim=-1) を current_beam_index で指定して抽出 -> (bsz, 1, 1)
            prob_sampled = prob_from_prev.gather(2, current_beam_index[:, :, None])
            
            sampled_log_probs_reversed.append(prob_sampled.squeeze(2)) # (bsz, 1)

        # 時刻 t=0 の確率を抽出して追加
        # finalized_tokens[-1] には、最初の時刻 t=0 で選ばれた最適パスのBeamインデックスが入っている
        t0_index = finalized_tokens[-1]
        prob_t0 = log_probs_t0.gather(1, t0_index) # (bsz, 1)
        sampled_log_probs_reversed.append(prob_t0)

        # 通常のトークンIDの復元処理
        finalized_tokens.reverse()
        finalized_tokens = torch.cat(finalized_tokens, 1)
        finalized_tokens = beam_targets.gather(2, finalized_tokens[:, :, None])[:, :, 0]

        finalized_scores.reverse()
        finalized_scores = torch.cat(finalized_scores, 1)
        finalized_scores[:, 1:] = finalized_scores[:, 1:] - finalized_scores[:, :-1]

        # 強化学習用の確率を時間順に戻して結合
        sampled_log_probs_reversed.reverse()
        # 形状: (bsz, seq_len, 1)
        sampled_log_probs = torch.cat(sampled_log_probs_reversed, dim=1)[:, :, None]

        return finalized_scores, finalized_tokens, sampled_log_probs




### CaptioningTransformer モデル定義

In [ ]:
class CaptioningTransformer(nn.Module):
    
    #img_size       :clip に入力する画像サイズ。
    #dim_embedding  :bert の隠れ次元
    #length_max     :固定長の seq_len。97。
    #vocab_size     :語彙数。
    #tokenizer      :tokenizer
    #dropout        :dropout の値
    #pad_token      :tokenizer.pad_token_id
    #use_repeat_logits_half: bert の出力の repeat している token の確率を小さくするか。 boolean
    #crf_coef       :crf_loss に乗ずる crf_coef。0 だと、crf_loss を計算しない。
    #temp           :_compute_grpo_samplesのmultinomial の前の softmax の温度。
    #top_p          :_compute_grpo_samples の top_p
    #top_k          :_compute_grpo_samples の top_k
    #num_samples    :_compute_grpo_samples の グループ数
    #ref_t          :model を refence_model とする場合 True。通常の場合 False
    
    def __init__(self, img_size: int,  dim_embedding: int, length_max: int, vocab_size: int, tokenizer, dropout: float = 0.0, \
                 pad_token_id: int=0, use_repeat_logits_half=False, crf_coef = 1.0, temp=0.5, top_p = 0.9, top_k = 50,
                 num_samples=16, crf_beam = 256, cand = 4, ref_t = False):
        super().__init__()

        #CLIP
        model_id = "openai/clip-vit-large-patch14-336"
        self.clip_model = CLIPVisionModel.from_pretrained(model_id )
        memory = self.clip_model( torch.randn( 1, 3, 336, 336 ) )
        memory = memory.last_hidden_state
        img_length = memory.size(1)
        clip_dim = memory.size(2)
        self.connector_pool = nn.AdaptiveAvgPool1d(length_max - 1 )
        self.connector_ln = nn.LayerNorm( clip_dim )
        self.connector_linear1 = nn.Linear( clip_dim, dim_embedding )
        self.connector_gleu = nn.GELU()
        self.connector_linear2 = nn.Linear( dim_embedding, dim_embedding )

       
        # Connector
        self.connector_pool = nn.AdaptiveAvgPool1d(length_max - 1 )
        # Down Sampling
        cls_token = memory[:, :1, :] # (bsz, 1, 1024)
        patch_tokens = memory[:, 1:, :] # (bsz, 576, 1024)
        # パッチ部分を 576 -> 96 に圧縮
        patch_tokens = patch_tokens.transpose(1, 2) # (bsz, 1024, 576)
        patch_tokens = self.connector_pool(patch_tokens)
        patch_tokens = patch_tokens.transpose(1, 2) # (bsz, 96, 1024)
        # CLSと結合して合計 97 トークンにする
        memory = torch.cat([cls_token, patch_tokens], dim=1) # (bsz, 97, 1024)

        self.pos_emb = PositionalEmbedding( dim_embedding )

        model_id = "google-bert/bert-large-uncased"
        self.bert = BertModel.from_pretrained( model_id )

        ## 単語出力分布計算
        self.ln_outputs = nn.LayerNorm( dim_embedding )
        self.linear = nn.Linear(dim_embedding, vocab_size)

        crf_low_rank = 32
        self.crf_beam_size = crf_beam
        self.cand = cand
        top_dropout = 0.0
        tgt_padding_idx = tokenizer.pad_token_id
        print( "initialize crf_layer" )
        self.ref_t = ref_t
        self.crf_layer = DynamicCRF(num_embedding = vocab_size, low_rank = crf_low_rank, beam_size = crf_beam, 
                                    crf_coef=crf_coef, temp=temp, top_p = top_p, top_k = top_k, num_samples= num_samples, 
                                    cand=self.cand, ref_t = ref_t )
        self.dim_embedding = dim_embedding
        self.use_repeat_logits_half = use_repeat_logits_half


    def mlp_connector(self, memory ):

        cls_token = memory[:, :1, :] # (bsz, 1, 1024)
        patch_tokens = memory[:, 1:, :] # (bsz, 576, 1024)

        # パッチ部分を 576 -> 96 に圧縮
        patch_tokens = patch_tokens.transpose(1, 2) # (bsz, 1024, 576)
        patch_tokens = self.connector_pool(patch_tokens)
        patch_tokens = patch_tokens.transpose(1, 2) # (bsz, 96, 1024)

        # CLSと結合して合計 97 トークンにする
        memory = torch.cat([cls_token, patch_tokens], dim=1) # (bsz, 97, 1024)

        memory = self.connector_ln( memory )
        memory = self.connector_linear1( memory )
        memory = self.connector_gleu( memory )
        memory = self.connector_linear2( memory )
        
        return memory

    def forward(self, images: torch.Tensor, targets: torch.Tensor, sampled_beam_idx = None, top_indices = None, use_crf_beam_logits = False, 
                grpo_mode = True, crf_mode = False ):

        self.device = images.device
        
        memory = self.clip_model( images ).last_hidden_state
        memory = self.mlp_connector( memory )
        memory += self.pos_emb( memory )
        
        outputs = self.bert( inputs_embeds = memory ).last_hidden_state
        outputs = self.ln_outputs( outputs )
        emissions = self.linear( outputs )
        
        if not self.ref_t:
            if grpo_mode:
                finalized_tokens, beam_probs, log_beam_probs, crf_loss, crf_beam_logits, top_indices, sampled_beam_idx, b_finalized_tokens = \
                    self.crf_layer(emissions, targets, top_indices = top_indices, sampled_beam_idx = sampled_beam_idx, use_crf_beam_logits = use_crf_beam_logits )
                return finalized_tokens, beam_probs, log_beam_probs, crf_loss, crf_beam_logits, emissions, top_indices, sampled_beam_idx, b_finalized_tokens
            else:
                if crf_mode:
                    finalized_tokens, crf_loss, crf_beam_logits, beam_targets = self.crf_layer( emissions, targets, grpo_mode = False, crf_mode = True , use_crf_beam_logits = use_crf_beam_logits)
                    return finalized_tokens, crf_loss, crf_beam_logits, beam_targets, emissions
                else:
                    finalized_tokens = self.crf_layer( emissions, targets, grpo_mode = False, use_crf_beam_logits = False ) 
                    return finalized_tokens
        else:
            ref_log_beam_probs, finalized_tokens, beam_targets = self.crf_layer(emissions, targets, top_indices = top_indices, 
                                                                           sampled_beam_idx = sampled_beam_idx )
            return ref_log_beam_probs, finalized_tokens, beam_targets
    
    def repeat_logits_half(self, emissions ):
        
        penalty = 2.0
        scores, preds = torch.max( emissions, 2 )
        masks = emissions == scores[:,:,None]
        masks = masks.permute( 1, 0, 2 )
        new_mask = torch.zeros( (  masks.size(1), masks.size(2)), device = emissions.device, dtype=torch.bool )
        new_masks = torch.zeros( ( masks.size(0), masks.size(1), masks.size(2)), device = emissions.device, dtype=torch.bool )
        for i, mask in enumerate( masks ):
            new_mask = torch.logical_or( mask,  new_mask  )
            new_masks[i] = new_mask
        new_masks = new_masks.transpose(0,1)
        first_true_mask = ( new_masks.int().cumsum(dim = 1 ) == 1 ) & new_masks
        new_masks = new_masks & ( ~first_true_mask )

        p_masks = emissions > 0
        m_masks = emissions < 0
        p_new_masks = p_masks & new_masks
        m_new_masks = p_masks & new_masks
        emissions2 = emissions.clone()
        emissions2[p_new_masks] = emissions[p_new_masks] / penalty
        emissions2[m_new_masks] = emissions2[m_new_masks] * penalty

        return emissions2


### 学習におけるハイパーパラメータやオプションの設定

In [ ]:
model_id = "google-bert/bert-large-uncased"
tokenizer = BertTokenizer.from_pretrained(model_id)
pad_token_id = tokenizer.pad_token_id
cls_token_id = tokenizer.cls_token_id
sep_token_id = tokenizer.sep_token_id
# 2. 新しい特殊トークンを登録
# 2. 新しい特殊トークンを登録
special_tokens_dict = {'additional_special_tokens': ['[unused0]']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
special_tokens_dict = {'additional_special_tokens': ['[unused1]']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
sos_token_id = tokenizer.encode( "[unused0]" )[1]
print( "sos_token_id:", sos_token_id )
eos_token_id = tokenizer.encode( "[unused1]" )[1]
print( "eos_token_id:", eos_token_id )
test = tokenizer.decode( sos_token_id )
print( test )
test = tokenizer.decode( eos_token_id )
print( test )
a_token_id = tokenizer.encode( "a"  )[1]
print( "a_token_id:", a_token_id )
an_token_id = tokenizer.encode( "an"  )[1]
the_token_id = tokenizer.encode( "the" )[1]
and_token_id = tokenizer.encode( "and" )[1]
in_token_id = tokenizer.encode( "in" )[1]
we_token_id = tokenizer.encode( "we" )[1]
i_token_id = tokenizer.encode( "i" )[1]
he_token_id = tokenizer.encode( "he" )[1]
she_token_id = tokenizer.encode( "she" )[1]
it_token_id = tokenizer.encode( "it" )[1]
they_token_id = tokenizer.encode( "they" )[1]
period_token_id = tokenizer.encode( "." )[1]
comma_token_id = tokenizer.encode( "," )[1]
dbl_token_id = tokenizer.encode( '"' )[1]
sgl_token_id = tokenizer.encode( "'" )[1]

# 辞書サイズを保存
vocab_size = len( tokenizer )


class ConfigTrain(object):
    '''
    ハイパーパラメータ、システム共通変数の設定
    '''
    def __init__(self):

        # ハイパーパラメータ
        self.img_size = 336
        self.dim_embedding = 1024   # 埋め込み層の次元
        self.length_max = 97
        #self.lr = 5e-5            # 学習率
        #self.lr = 2e-5            # 学習率
        self.lr_clip = 0
        self.lr_con = 1000
        self.lr_bert = 1000            # 学習率
        self.lr_crf = 0
        self.lr_others = 1000
        #self.lr_top = 1e-4
        #self.lr = 5e-6            # 学習率
        self.clip_grad_threshold = 1000
        self.dropout = 0.2         # dropout確率
        #self.batch_size = 128       # ミニバッチ数
        self.batch_size = 80
        #self.batch_size = 64       # ミニバッチ数
        #self.batch_size = 40       # ミニバッチ数
        #self.batch_size = 32       # ミニバッチ数
        #self.batch_size = 24       # ミニバッチ数
        #self.batch_size = 16       # ミニバッチ数
        #self.batch_size = 8       # ミニバッチ数
        #self.batch_size = 4       # ミニバッチ数
        #self.batch_size = 2
        #self.batch_size = 1       # ミニバッチ数
        #self.num_epochs = 100       # エポック数→Colab無料版でテストする際は10未満に修正を推奨
        #self.num_epochs = 100       # エポック数→Colab無料版でテストする際は10未満に修正を推奨
        #self.num_epochs = 60       # エポック数→Colab無料版でテストする際は10未満に修正を推奨
        self.num_epochs = 1       # エポック数→Colab無料版でテストする際は10未満に修正を推奨
        self.use_amp = True
        #self.use_amp = False
        self.use_saved_pth = True
        #self.use_saved_pth = False
        self.model_id = "google-bert/bert-large-uncased"
        self.vocab_size = len( tokenizer )
        self.weight_decay = 1000
        self.betas = (0.9, 0.999 )
        self.warmup = 0.1
        #self.alpha = 1.0
        self.crf_coef = 1000
        self.ce_coef = 1000
        self.use_repeat_logits_half = False
        self.temp = 1.0
        self.top_p = 1.0
        self.top_k = 0
        self.num_samples = 0
        self.beam = 256
        self.cand = 4
        self.use_crf_beam_logits = True
        self.use_captions_beam = True # True use captions_beam when calculate crf_loss
        self.early_stopping_count_thresh = 4
        self.early_stopping_threshold_thresh = 5
        self.cider_thresh = 0.6
        self.train_param = 50
        self.val_param = 50
        self.val_interval = 10
         
        # パスの設定
        #self.img_directory = '/mnt/ssd1/uchiyats/python_image_recognition-main/6_img_captioning/6_5_myoriginal_transformer_captioning/train2017'
        #self.anno_file = '/mnt/ssd1/uchiyats/python_image_recognition-main/data/coco2014/captions_train2017.json'
        self.img_directory = './train2017'
        self.anno_file = './data/captions_train2017.json'
        self.save_directory = './model'
        self.PATH = "../pre_train_crf/model/model_ImgCap_SFT_COCO_curr.pth"
        #self.img_directory = 'J:\\python_image_recognition-main\\6_img_captioning\\6_5_myoriginal_transformer_captioning\\train2017'
        #self.anno_file = 'J:\\python_image_recognition-main\\data\\coco2014\\captions_train2017.json'
        #self.save_directory = './model'
        #self.PATH = "J:\\RL\\GRPO2\\COCO\\model\\model_ImgCap_SFT_COCO_2_curr.pth"
        #self.PATH = 'J:\\RL\\GRPO2\\pre_train_crf\\model\\model_COCO_tr5_weights.pth'

        
        # 検証に使う学習セット内のデータの割合
        #self.test_ratio = 0.0005
        #self.val_ratio = 0.0005
        #self.val_ratio = 0.004
        #self.test_ratio = 0.004
        self.test_ratio = 0.01
        self.val_ratio = 0.01
        
        # 学習に使うデバイス
        #self.device = 'cuda'
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        #self.device = 'cpu'
        
        # データローダーに使うCPUプロセスの数
        #self.num_workers = 4
        self.num_workers = 0 if self.device == torch.device('cpu') else 12
        #self.num_workers = 0
        
        # 移動平均で計算する損失の値の数
        self.moving_avg = 100

### 学習を行う関数

In [ ]:
def objective(trial):

    config.lr_con = trial.suggest_float("config.lr_con", 1e-8,1e-4,log=True)
    config.lr_bert = trial.suggest_float("config.lr_bert", 1e-8,1e-4,log=True)
    #config.lr_crf = trial.suggest_float("config.lr_crf", 1e-10, 1e-6, log=True)
    config.lr_others = trial.suggest_float("config.lr_others", 1e-8, 1e-4, log=True)
    config.weight_decay = trial.suggest_float("config.weight_decay", 1e-6,1e-1,log=True)
    config.clip_grad_threshold = trial.suggest_float("config.clip_grad_threshold", 0, 5 )
    config.crf_coef = trial.suggest_float("config.crf_coef", 0.0,1.0)
    config.ce_coef = trial.suggest_float("config.ce_coef", 0.0,1.0)

    params = {
        'config.lr_con': config.lr_con, 'config.lr_bert': config.lr_bert,
        'config.lr_others': config.lr_others, 
        'config.weight_decay': config.weight_decay, 
        'config.clip_grad_threshold': config.clip_grad_threshold,
        'config.crf_coef': config.crf_coef, 'config.ce_coef': config.ce_coef,
    }
        #'config.ord_coef': config.ord_coef,

    
    # モデルの定義
    model = CaptioningTransformer( config.img_size,
        config.dim_embedding, config.length_max, config.vocab_size,
        tokenizer, config.dropout, pad_token_id = tokenizer.pad_token_id,
        use_repeat_logits_half = config.use_repeat_logits_half,
        crf_coef = config.crf_coef, temp=config.temp, top_p = config.top_p, 
        top_k = config.top_k, num_samples=config.num_samples, crf_beam = config.beam, cand = config.cand )
    model.to(config.device)
    
    
    # 損失関数の定義
    #criterion = nn.CrossEntropyLoss( ignore_index = tokenizer.pad_token_id, reduction = 'mean' )
    #criterion = nn.CrossEntropyLoss( reduction = 'mean' )
    #log_softmax = nn.LogSoftmax( dim = 2 )
    #softmax = nn.Softmax( dim = 2 )
    #criterion_nll = nn.NLLLoss( reduction = 'none' )
    #criterion_nll = nn.NLLLoss( )
    #criterion = nn.CrossEntropyLoss()
    
    # 最適化手法の定義
    # 最適化手法の定義
    # Optimizerの生成, clipとそうでないモジュールとの
    # パラメータで異なる学習率を適用
    #params_clip = []
    params_con = []
    params_bert = []
    #params_crf = []
    params_others = []
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            if 'clip_model' in name:
                #params_clip.append(parameter)
                parameter.requires_grad = False
            elif 'connector' in name:
                params_con.append(parameter)
                #parameter.requires_grad = False
            elif 'bert' in name and 'critical' not in name:
                params_bert.append(parameter)
                #parameter.requires_grad = False
            elif 'crf_layer' in name:
                #params_crf.append(parameter)
                parameter.requires_grad = False
            else:
                params_others.append(parameter)
    param_groups = [
        #{'params': params_clip, 'lr': config.lr_clip},
        {'params': params_con, 'lr': config.lr_con},
        {'params': params_bert, 'lr': config.lr_bert},
        #{'params': params_crf, 'lr': config.lr_crf},
        {'params': params_others, 'lr': config.lr_others}]
    #optimizer = torch.optim.AdamW( model.parameters() , lr=config.lr)
    #optimizer = torch.optim.AdamW( param_groups, weight_decay = config.weight_decay, betas=config.betas )
    optimizer = torch.optim.AdamW( param_groups, weight_decay = config.weight_decay, betas=config.betas )
    
    # 全ステップ数
    num_global_steps = len( train_loader ) * config.num_epochs
    print( "num_global_steps:", num_global_steps )
    num_warmup_steps = num_global_steps * config.warmup
    print( "num_warmup_steps:", num_warmup_steps )
    #スケジューラーの定義
    scheduler = get_linear_schedule_with_warmup( optimizer, num_warmup_steps, num_global_steps )    
    #t_scheduler = get_linear_schedule_with_warmup( t_optimizer, num_warmup_steps, num_global_steps )    
    
    
    #PATH = "../pre_train_crf/model/model_ImgCap_SFT_COCO_curr.pth"
    PATH = config.PATH
    print( "use_saved_pth:", config.use_saved_pth )
    print( "exist saved_pth:", os.path.isfile(PATH) ) 
    use_saved_pth = config.use_saved_pth
    if use_saved_pth and os.path.isfile(PATH):
        checkpoint = torch.load(PATH, map_location='cpu')
        model.load_state_dict(checkpoint['model_state_dict'])
        print( "loaded model parameters." )
        #checkpoint = torch.load(config.PATH, map_location=torch.device('cpu'), weights_only=True)
        #model.load_state_dict(checkpoint, strict = False)
        #print( "model parameters were loaded")
        #optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        #scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        #device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        ## optimizerのstateを現在のdeviceに移す。これをしないと、保存前後でdeviceの不整合が起こる可能性がある。
        #for state in optimizer.state.values():
            #for k, v in state.items():
                #if isinstance(v, torch.Tensor):
                    #state[k] = v.to(device)
        #begin_epoch = checkpoint['epoch']
        #loss = checkpoint['loss']
        #global_step = checkpoint['global_step']    
    else:
        begin_epoch = 0
        global_step = 0
    
    
    begin_epoch = 0
    global_step = 0
    val_global_step = 0
    
    
    print( "begin_epoch:", begin_epoch )
    print( "global_step:", global_step )
    
    len_tr_loader = len( train_loader )
    #train_param = len_tr_loader // 100
    #train_param = 100
    #train_param = len_tr_loader // 6
    len_val_loader = len( val_loader )
    #train_param = len_val_loader // 3
    #val_param = len_train_loader // 3
    #val_param = 100
    print( "train_param:", config.train_param )
    print( "val_param:", config.val_param )
    print( "val_interval:", config.val_interval )
    
    print( "epochs:", config.num_epochs )
    print( "batch_size:", config.batch_size )
    print( "lr_clip:", config.lr_clip )
    print( "lr_con:", config.lr_con )
    print( "lr_bert:", config.lr_bert )
    print( "lr_others:", config.lr_others )
    print( "weight_decay:", config.weight_decay )
    print( "betas:", config.betas )
    print( "crf_coef:", config.crf_coef )
    print( "ce_coef:", config.ce_coef )
    print( "use_repeat_logits_half:", config.use_repeat_logits_half )
    print( "use_crf_beam_logits:", config.use_crf_beam_logits )
    print( "use_captions_beam:", config.use_captions_beam )
    
    # 学習経過の書き込み
    now = datetime.datetime.now()
    train_loss_file = '{}/MyOriginal_train_loss_{}.csv'\
        .format(config.save_directory, now.strftime('%Y%m%d_%H%M%S'))
    with open(train_loss_file, 'a') as f:
        print(f'{len_tr_loader}', file=f) 
    print( "train_loss_file:", train_loss_file )
    val_loss_file = '{}/MyOriginal_val_loss_{}.csv'\
        .format(config.save_directory, now.strftime('%Y%m%d_%H%M%S'))
    with open(val_loss_file, 'a') as f:
        print(f'{len_val_loader}', file=f) 
    
    
    fn = SmoothingFunction().method7
    #rougeN = rouge_scorer.RougeScorer(['rouge1', 'rouge2'], use_stemmer=True) 
    rougeL = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    bert_score_fn = load('bertscore')
    cider = Cider()
    clip_score_fn = CLIPScore(model_name_or_path="openai/clip-vit-base-patch32").to(config.device)
    
    # AMP用のスケーラー
    scaler = GradScaler(enabled=config.use_amp)
    
    pad_token_id = tokenizer.pad_token_id
    
    #last_rapid_early_stopping_count = 0
    early_stopping_count = 0
    last_avg_bleu = float( '-inf' )
    last_avg_bert = float( '-inf' )
    last_avg_cider = float( '-inf' )
    val_cider_best = float('-inf')
    val_cider_best_step = 0
    early_stopping_threshold = 0
    
    for epoch in range(config.num_epochs):
        with tqdm(train_loader) as pbar:
        #with tqdm(val_loader) as pbar:
            pbar.set_description(f'[エポック {epoch + 1}]')
    
            # 学習モードに設定
            model.train()
    
            train_losses = deque()
            train_crfs = deque()
            train_ces = deque()
            #train_bleus = deque()
            #train_ciders = deque()
            #train_rouges = deque()
            #train_berts = deque()
            for n_batch, (imgs, imgs2, captions, caption_lengths) in enumerate( pbar ):
                # ミニバッチを設定
                imgs = imgs.to(config.device)
                imgs2 = imgs2.to(config.device )
                captions = captions.to(config.device)
                    
                optimizer.zero_grad()
    
                # 最後の単語から次を予測する必要はないため最後の単語を除外
                with autocast(str(config.device),enabled=config.use_amp):
    
                    finalized_tokens, crf_loss, crf_beam_logits, beam_targets, bert_emissions = \
                        model( imgs, captions, top_indices = None, grpo_mode = False, crf_mode = True, 
                               use_crf_beam_logits = config.use_crf_beam_logits )
    
                    if config.use_crf_beam_logits:
    
                        if not config.use_captions_beam:
                            # 1. ターゲットがビーム内に含まれているかチェックするマスクを作成
                            # beam_targets: (bsz, seq_len, beam)
                            # captions: (bsz, seq_len)
                            is_in_beam = (beam_targets == captions.unsqueeze(-1)).any(dim=-1)
                            
                            # 2. ビーム外の正解ラベルを一時的に無視用のインデックス（例: -100）に置換
                            # これにより、ビーム外の単語の -inf を参照して損失が inf になるのを防ぎます
                            IGNORE_IDX = -100
                            #IGNORE_IDX = 0
                            masked_captions = torch.where(is_in_beam, captions, torch.tensor(IGNORE_IDX, device=captions.device))
        
                            crf_logits = torch.full( ( crf_beam_logits.size(0), crf_beam_logits.size(1), vocab_size ), float('-inf'), 
                                                        dtype=crf_beam_logits.dtype, device = config.device )
                            
                            # 4. ignore_index を指定して Loss を計算
                            # これにより、ビーム内の予測のみで正しく微分可能なバックプロパゲーションが行われます
                            criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_IDX)
                            a = config.crf_coef * criterion(crf_logits.transpose(1, 2), masked_captions)
        
                            #crf_logits = torch.full( ( crf_beam_logits.size(0), crf_beam_logits.size(1), vocab_size ), float('-inf'), 
                            #                            dtype=crf_beam_logits.dtype, device = config.device )
                            #crf_logits = torch.scatter( crf_logits, -1, index = beam_targets, src = crf_beam_logits )
                            #a = config.crf_coef * nn.CrossEntropyLoss()( crf_logits.transpose(1,2), captions )
    
                        else:                    
                            ## vocab_size のfinalized_tokens から beam の sampled_beam_idx を作る
                            ## 2. ビーム次元 (-1) で一致しているインデックスを取得
                            #mask = beam_targets == captions.unsqueeze(-1) #B,S,W
                            #captions_beam = torch.argmax(mask.to(torch.int32), dim=-1)
                            #a = config.crf_coef * nn.CrossEntropyLoss()( crf_beam_logits.transpose(1,2), captions_beam )
        
                            # 1. 一致する場所を判定
                            mask = (beam_targets == captions.unsqueeze(-1))  # 形状: (B, S, beam)
                            
                            # 2. ビーム内に正解が含まれているかどうかのフラグ
                            is_in_beam = mask.any(dim=-1)  # 形状: (B, S)
                            #print( "is_in_beam:", is_in_beam )
                            
                            # 3. 一旦 argmax でインデックスを取得
                            captions_beam = torch.argmax(mask.to(torch.int32), dim=-1)  # 形状: (B, S)
                            
                            # 4. ビーム外のトークンは無視用インデックス（-100）に置き換える
                            IGNORE_IDX = -100
                            #IGNORE_IDX = 0
                            captions_beam = torch.where(is_in_beam, captions_beam, torch.tensor(IGNORE_IDX, device=captions.device))
                            
                            # 5. ignore_index を指定して損失を計算
                            # (crf_beam_logits の形状は (B, S, beam) なので transpose(1,2) で (B, beam, S) にする)
                            criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_IDX)
                            a = config.crf_coef * criterion(crf_beam_logits.transpose(1, 2), captions_beam)
                        
                    else:
                        a = config.crf_coef * crf_loss
                    
                    b = config.ce_coef * nn.CrossEntropyLoss()( bert_emissions.transpose(1,2), captions )
                    loss = a + b
    
                train_losses.append(loss.item())
                train_crfs.append(a.item())
                train_ces.append(b.item())
    
                if len(train_losses) > config.moving_avg:
                    train_losses.popleft()
                    train_crfs.popleft()
                    train_ces.popleft()
    
                
                hypo_ids = finalized_tokens
                
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(\
                        model.parameters(),
                        config.clip_grad_threshold)
                # オプティマイザにより，パラメータを更新する
                scaler.step(optimizer)
                scaler.update()            
                scheduler.step()
    
                global_step += 1
    
                if n_batch == 0 or n_batch % config.train_param == ( config.train_param - 1 ) or n_batch == len( train_loader ) - 1:
                    
                    preds1 = hypo_ids.clone().to(config.device)
                    preds2 = hypo_ids.clone().to(config.device )
                    targets1 = captions.clone().to(config.device)
                    
                    # 2. リスト内包表記の構文エラーを修正
                    preds_decoded = tokenizer.batch_decode(preds1, skip_special_tokens=False)
                    preds_str = [s.replace("[PAD]", "").strip() for s in preds_decoded]
                    
                    targets_decoded = tokenizer.batch_decode(targets1, skip_special_tokens=False)
                    targets_str = [s.replace("[PAD]", "").strip() for s in targets_decoded]
            
                   
                    # 5. BLEUの入力形式をトークン単位・参照文ネスト構造に変換
                    # 参照文 (references) は [[['単語', '単語']], [['単語']]] の3次元構造が必要
                    tokenized_targets = [[target.split()] for target in targets_str]
                    tokenized_preds = [pred.split() for pred in preds_str]
                    
                    # 変数名の競合を防ぐためスコアは別名で保存
                    bleu_score = corpus_bleu(tokenized_targets, tokenized_preds, smoothing_function=fn)        
        
                    pred_dict = { str(i): [item] for i, item in enumerate( preds_str)}
                    target_dict = { str(i): [item] for i, item in enumerate( targets_str)}
                    score, scores = cider.compute_score(target_dict, pred_dict)
                    cider_score = torch.tensor( scores ).to( config.device )
        
                    # 3. ROUGEスコアのキーを 'rouge2' に修正 (fmeasureは[2]または.fmeasure)
                    rouge_score = [[rougeL.score(target, pred)['rougeL'].fmeasure] for pred, target in zip(preds_str, targets_str)]
                    rouge_score = torch.tensor(rouge_score).to(config.device)
    
                    preds2[preds2 == eos_token_id] = pad_token_id
                    preds_str2 = tokenizer.batch_decode(preds2, skip_special_tokens = True )
                    
                    with torch.no_grad():
                        processed = clip_score_fn.processor(text=preds_str2, images=imgs2, return_tensors="pt", padding=True, \
                                                          truncation=True, max_length=77, do_resize=False, do_rescale=False ).to(config.device)
                        outputs = clip_score_fn.model(**processed)
                        # 特徴量の正規化
                        image_features = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
                        text_features = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)
                        individual_scores = torch.clamp( (image_features.to(config.device) * \
                                                          text_features.to(config.device)).sum(axis=-1), min=0)
                        clip_score = individual_scores
                    
                    preds_str = [p if p.strip() != "" else "." for p in preds_str]
                    targets_str = [t if t.strip() != "" else "." for t in targets_str]
                    
                    ## 4. BERTScore の計算
                    with torch.no_grad():
                        model_name = 'distilbert-base-uncased'
                        bert_scores = bert_score_fn.compute(
                            predictions=preds_str,
                            references=targets_str,
                            model_type=model_name,
                            use_fast_tokenizer=False,
                            lang='en',
                            device=config.device,
                            batch_size=1,
                            rescale_with_baseline=False
                        )['f1']
                        
                    bert_score = torch.tensor(bert_scores).to(config.device)
        
                    bleu_score_mean = bleu_score
                    cider_score_mean = cider_score.mean().item()
                    rouge_score_mean = rouge_score.mean().item()
                    bert_score_mean = bert_score.mean().item()
                    clip_score_mean = clip_score.mean().item()
    
                    #train_bleus.append(bleu_score_mean)
                    #train_ciders.append(cider_score_mean)
                    #train_rouges.append(rouge_score_mean)
                    #train_berts.append(bert_score_mean)
                    #if len(train_losses) > config.moving_avg:
                    #    train_losses.popleft()
                    #    train_crfs.popleft()
                    #    train_ces.popleft()
                    #    train_bleus.popleft()
                    #    train_ciders.popleft()
                    #    train_rouges.popleft()
                    #    train_berts.popleft()
                    mean_loss = torch.Tensor(train_losses).mean().item()
                    mean_crf = torch.Tensor(train_crfs).mean().item()
                    mean_ce = torch.Tensor(train_ces).mean().item()
                    #mean_bleu = torch.Tensor(train_bleus).mean().item()
                    #mean_cider = torch.Tensor(train_ciders).mean().item()
                    #mean_rouge = torch.Tensor(train_rouges).mean().item()
                    #mean_bert = torch.Tensor(train_berts).mean().item()
                    pbar.set_postfix({
                        'loss': mean_loss,
                        'crf': mean_crf,
                        'ce': mean_ce,
                        'bleu': bleu_score_mean,
                        'cider': cider_score_mean,
                        'rouge': rouge_score_mean,
                        'bert': bert_score_mean,
                        'clip': clip_score_mean,
                        #'CIDER': torch.Tensor(train_ciders).mean().item()
                    })
                    with open(train_loss_file, 'a') as f:
                        print(f'{global_step }, {mean_loss}, {mean_crf}, {mean_ce}, {bleu_score_mean}, {cider_score_mean},',\
                              f'{rouge_score_mean}, {bert_score_mean}, {clip_score_mean}', file=f)
                    #if n_batch == 0 or n_batch % 50 == 49 or n_batch == len( train_loader ) - 1:
                    #print( "lr clip     :", optimizer.param_groups[0]["lr"] )
                    print( "\nTrain phase." )
                    print( "lr connector:", optimizer.param_groups[0]["lr"] )
                    print( "lr bert     :", optimizer.param_groups[1]["lr"] )
                    #print( "lr crf      :", optimizer.param_groups[1]["lr"] )
                    print( "lr others   :", optimizer.param_groups[2]["lr"] )
                    print(f'global step = {global_step }, loss = {mean_loss}, crf = {mean_crf}, ce = {mean_ce}, bleu = {bleu_score_mean}, ', \
                          f'cider = {cider_score_mean}, rouge = {rouge_score_mean}, bert = {bert_score_mean}, clip = {clip_score_mean}')
                    print( "refe:", targets_str[0] )
                    print( "hypo:", preds_str[0] )
                    
    
                if n_batch == 0 or n_batch % config.val_param == ( config.val_param - 1 ) or n_batch == len( train_loader ) - 1:
                    # 検証
    
                    print( "\nValidation phase." )
                    with tqdm(val_loader) as pbar:
                        pbar.set_description(f'[検証]')
                
                        # 評価モード
                        model.eval()
                
                        #val_losses = []
                        #val_losses = deque()
                        #val_a = deque()
                        #val_b = deque()
                        val_bleus = deque()
                        val_ciders = deque()
                        val_rouges = deque()
                        val_berts = deque()
                        val_clips = deque()
                        for val_n_batch, (imgs, imgs2, captions, caption_lengths) in enumerate( pbar ):
                
                            # ミニバッチを設定
                            imgs = imgs.to(config.device)
                            imgs2 = imgs2.to(config.device )
                            captions = captions.to(config.device)
                            #caption_lengths = torch.tensor( caption_lengths ).to(config.device)
                                
                            with torch.no_grad():
                                finalized_tokens, crf_loss, crf_beam_logits, beam_targets, bert_emissions = \
                                    model( imgs, captions, top_indices = None, grpo_mode = False, crf_mode = True )
                
                            hypo_ids = finalized_tokens
                               
                            preds1 = hypo_ids.clone().to(config.device)
                            preds2 = hypo_ids.clone().to(config.device )
                            targets1 = captions.clone().to(config.device)
                            
                            # 2. リスト内包表記の構文エラーを修正
                            preds_decoded = tokenizer.batch_decode(preds1, skip_special_tokens=False)
                            preds_str = [s.replace("[PAD]", "").strip() for s in preds_decoded]
                            
                            targets_decoded = tokenizer.batch_decode(targets1, skip_special_tokens=False)
                            targets_str = [s.replace("[PAD]", "").strip() for s in targets_decoded]
                    
                           
                            # 5. BLEUの入力形式をトークン単位・参照文ネスト構造に変換
                            # 参照文 (references) は [[['単語', '単語']], [['単語']]] の3次元構造が必要
                            tokenized_targets = [[target.split()] for target in targets_str]
                            tokenized_preds = [pred.split() for pred in preds_str]
                            
                            # 変数名の競合を防ぐためスコアは別名で保存
                            bleu_score = corpus_bleu(tokenized_targets, tokenized_preds, smoothing_function=fn)        
                
                            pred_dict = { str(i): [item] for i, item in enumerate( preds_str)}
                            target_dict = { str(i): [item] for i, item in enumerate( targets_str)}
                            score, scores = cider.compute_score(target_dict, pred_dict)
                            cider_score = torch.tensor( scores ).to( config.device )
                
                           
                            # 3. ROUGEスコアのキーを 'rouge2' に修正 (fmeasureは[2]または.fmeasure)
                            rouge_score = [[rougeL.score(target, pred)['rougeL'].fmeasure] for pred, target in zip(preds_str, targets_str)]
                            rouge_score = torch.tensor(rouge_score).to(config.device)
                            
                            preds2[preds2 == eos_token_id] = pad_token_id
                            preds_str2 = tokenizer.batch_decode(preds2, skip_special_tokens = True )
                            
                            with torch.no_grad():
                                processed = clip_score_fn.processor(text=preds_str2, images=imgs2, return_tensors="pt", padding=True, \
                                                                  truncation=True, max_length=77, do_resize=False, do_rescale=False ).to(config.device)
                                outputs = clip_score_fn.model(**processed)
                                # 特徴量の正規化
                                image_features = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
                                text_features = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)
                                individual_scores = torch.clamp( (image_features.to(config.device) * \
                                                                  text_features.to(config.device)).sum(axis=-1), min=0)
                                clip_score = individual_scores
    
                            preds_str = [p if p.strip() != "" else "." for p in preds_str]
                            targets_str = [t if t.strip() != "" else "." for t in targets_str]
                            
                            ## 4. BERTScore の計算
                            with torch.no_grad():
                                model_name = 'distilbert-base-uncased'
                                bert_scores = bert_score_fn.compute(
                                    predictions=preds_str,
                                    references=targets_str,
                                    model_type=model_name,
                                    use_fast_tokenizer=False,
                                    lang='en',
                                    device=config.device,
                                    batch_size=1,
                                    rescale_with_baseline=False
                                )['f1']
                                
                            bert_score = torch.tensor(bert_scores).to(config.device)
                            #reward_bert = torch.zeros( (1) )
                
                            bleu_score_mean = bleu_score
                            cider_score_mean = cider_score.mean().item()
                            rouge_score_mean = rouge_score.mean().item()
                            bert_score_mean = bert_score.mean().item()
                            clip_score_mean = clip_score.mean().item()
                            #total_bleu = total_bleu + bleu_score_mean
                            #total_bert = bert_score_mean
                            #total_cider = cider_score_mean
                            
                            #val_losses.append(loss.item())
                            #val_crfs.append(a.item())
                            #val_cas.append(b.item())
                            val_bleus.append(bleu_score_mean)
                            val_ciders.append(cider_score_mean)
                            val_rouges.append(rouge_score_mean)
                            val_berts.append(bert_score_mean)
                            val_clips.append(clip_score_mean)
                            if len(val_bleus) > config.moving_avg:
                                #val_losses.popleft()
                                #val_crfs.popleft()
                                #val_ces.popleft()
                                val_bleus.popleft()
                                val_ciders.popleft()
                                val_rouges.popleft()
                                val_berts.popleft()
                                val_clips.popleft()
                            val_global_step = val_global_step + 1
                            #mean_loss = torch.Tensor(train_losses).mean().item()
                            #mean_crf = torch.Tensor(train_crfs).mean().item()
                            #mean_ce = torch.Tensor(train_ces).mean().item()
                            mean_bleu0 = torch.Tensor(val_bleus).mean().item()
                            mean_cider0 = torch.Tensor(val_ciders).mean().item()
                            mean_rouge0 = torch.Tensor(val_rouges).mean().item()
                            mean_bert0 = torch.Tensor(val_berts).mean().item()
                            mean_clip0 = torch.Tensor(val_clips).mean().item()
                            pbar.set_postfix({
                                #'loss': mean_loss,
                                #'crf': mean_crf,
                                #'ca': mean_ce,
                                'bleu': mean_bleu0,
                                'cider': mean_cider0,
                                'rouge': mean_rouge0,
                                'bert': mean_bert0,
                                'clip': mean_clip0,
                                #'CIDER': torch.Tensor(train_ciders).mean().item()
                            })
    
                            if val_n_batch == 0 or val_n_batch % config.val_interval == ( config.val_interval - 1 ) or val_n_batch == len( val_loader ) - 1:
                                print(f'val global_step = {val_global_step }, bleu = {mean_bleu0}, cider = {mean_cider0}, ',\
                                      f'rouge = {mean_rouge0}, bert = {mean_bert0}, clip = {mean_clip0}')
                                print( "refe:", targets_str[0] )
                                print( "hypo:", preds_str[0] )
                        
                    #mean_loss = torch.Tensor(train_losses).mean().item()
                    #mean_crf = torch.Tensor(train_crfs).mean().item()
                    #mean_ce = torch.Tensor(train_ces).mean().item()
                    mean_bleu1 = torch.Tensor(val_bleus).mean().item()
                    mean_cider1 = torch.Tensor(val_ciders).mean().item()
                    mean_rouge1 = torch.Tensor(val_rouges).mean().item()
                    mean_bert1 = torch.Tensor(val_berts).mean().item()
                    mean_clip1 = torch.Tensor(val_clips).mean().item()
    
                    with open(val_loss_file, 'a') as f:
                        print(f'{val_global_step}, {mean_bleu1}, {mean_cider1}, {mean_rouge1}, {mean_bert1}, {mean_clip1}', file=f)
    
                    #avg_bleu = total_bleu / len( val_loader )
                    #avg_bert = total_bert / len( val_loader )
                    #avg_cider = total_cider / len( val_loader )
                    avg_bleu = mean_bleu1
                    avg_bert = mean_bert1
                    avg_cider = mean_cider1
                    early_stopping_threshold = early_stopping_threshold + 1
                    print( "\n判定のための値。" )
                    print( "val_global_step:", val_global_step )
                    print( "avg_bleu:", avg_bleu, "  last_avg_bleu:", last_avg_bleu )
                    print( "avg_bert:", avg_bert, "  last_avg_bert:", last_avg_bert )
                    print( "avg_cider:", avg_cider, "  last_avg_cider:", last_avg_cider )
                    print( "early_stopping_count:", early_stopping_count, "  early_stopping_thershold:", early_stopping_threshold )
                    #if last_avg_bleu > avg_bleu:
                    #if last_avg_bert > avg_bert:
                    if last_avg_cider > avg_cider:
                        early_stopping_count = early_stopping_count + 1
                        #print( "avg_bleu < last_avg_bleu:" )
                        #print( "avg_bert < last_avg_bert:" )
                        print( "avg_cider < last_avg_cider:" )
                        print( "early_stopping_count:", early_stopping_count, "  early_stopping_thershold:", early_stopping_threshold )
                        if avg_cider < config.cider_thresh or  (early_stopping_count >= config.early_stopping_count_thresh and early_stopping_threshold >=  config.early_stopping_threshold_thresh ):
                            if avg_cider < config.cider_thresh:
                                print( f'avg_cider < {config.cider_thresh}. break.' )
                            else:
                                print( "early_stopping_count exceed. break." )
                            break
                    else:
                        early_stopping_count = 0
                        if avg_cider >= val_cider_best:
                            val_cider_best = avg_cider
                            val_cider_best_step = global_step
                    
                    last_avg_bleu = avg_bleu
                    last_avg_bert = avg_bert
                    last_avg_cider = avg_cider

                    # 4. 中間報告 (枝刈り用)
                    trial.report(float(avg_cider), global_step)
                    if trial.should_prune():
                        raise optuna.exceptions.TrialPruned()
            
            if  avg_cider < config.cider_thresh or  (early_stopping_count >= config.early_stopping_count_thresh and early_stopping_threshold >=  config.early_stopping_threshold_thresh ):
                if avg_cider < config.cider_thresh:
                    print( f'avg_cider < {config.cider_thresh}. break.' )
                else:
                    print( "early_stopping_count exceed. break." )
                print( "val_cider_best:", val_cider_best )
                break
            
    # 2. 損失と報酬の計算
    # 実際には学習データが必要ですが、ここではダミー
    print( f'global_step = { global_step }, avg_cider = {avg_cider}' )
    print( f'val_cider_best = {val_cider_best}, val_cider_best_step = { val_cider_best_step }' )
    
     # Optunaは目的関数が「最大化」か「最小化」かを選択する。
    # ここでは報酬を最大化したいので、負の報酬を返すか、direction="maximize"を指定する
    return val_cider_best

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

#torch.autograd.set_detect_anomaly(True, check_nan=False)

#os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'true'
config = ConfigTrain()

model_id = "google-bert/bert-large-uncased"
tokenizer = BertTokenizer.from_pretrained(model_id)
pad_token_id = tokenizer.pad_token_id
cls_token_id = tokenizer.cls_token_id
sep_token_id = tokenizer.sep_token_id
# 2. 新しい特殊トークンを登録
# 2. 新しい特殊トークンを登録
special_tokens_dict = {'additional_special_tokens': ['[unused0]']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
special_tokens_dict = {'additional_special_tokens': ['[unused1]']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
sos_token_id = tokenizer.encode( "[unused0]" )[1]
print( "sos_token_id:", sos_token_id )
eos_token_id = tokenizer.encode( "[unused1]" )[1]
print( "eos_token_id:", eos_token_id )
test = tokenizer.decode( sos_token_id )
print( test )
test = tokenizer.decode( eos_token_id )
print( test )
a_token_id = tokenizer.encode( "a"  )[1]
print( "a_token_id:", a_token_id )
an_token_id = tokenizer.encode( "an"  )[1]
the_token_id = tokenizer.encode( "the" )[1]
and_token_id = tokenizer.encode( "and" )[1]
in_token_id = tokenizer.encode( "in" )[1]
we_token_id = tokenizer.encode( "we" )[1]
i_token_id = tokenizer.encode( "i" )[1]
he_token_id = tokenizer.encode( "he" )[1]
she_token_id = tokenizer.encode( "she" )[1]
it_token_id = tokenizer.encode( "it" )[1]
they_token_id = tokenizer.encode( "they" )[1]
period_token_id = tokenizer.encode( "." )[1]
comma_token_id = tokenizer.encode( "," )[1]
dbl_token_id = tokenizer.encode( '"' )[1]
sgl_token_id = tokenizer.encode( "'" )[1]

# 辞書サイズを保存
vocab_size = len( tokenizer )

# モデル出力用のディレクトリを作成
os.makedirs(config.save_directory, exist_ok=True)

# 画像のtransformsを定義
transforms = v2.Compose([
    v2.Resize((336, 336)),
    v2.AutoAugment(),
    #v2.ToTensor(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    ## Coco データセット 2017 train の平均と標準偏差
    #v2.Normalize((0.456,0.427,0.401),(0.224,0.219,0.231) )
    # ImageNetデータセットの平均と標準偏差
    #v2.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
    # Clip Model の config から引用。
    v2.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))
])

# COCOデータロードの定義
train_dataset = dataset.CocoCaptions(root=config.img_directory,
                        annFile=config.anno_file,
                        transform=transforms)

# Subset samplerの生成
#val_set, train_set = util.generate_subset(
#    train_dataset, config.val_ratio)
test_set, val_set, train_set = util.generate_subset_test_val_train(
    train_dataset, config.test_ratio, config.val_ratio )
    
# 学習時にランダムにサンプルするためのサンプラー
train_sampler = SubsetRandomSampler(train_set)

# DataLoaderを生成
#collate_func_lambda = lambda x: util.collate_func(x, tokenizer, eos_token_id, pad_token_id, config.length_max )
collate_func_lambda = lambda x: util.collate_func5(x, tokenizer, eos_token_id, pad_token_id, config.length_max)
train_loader = torch.utils.data.DataLoader(
                    train_dataset,
                    batch_size=config.batch_size,
                    num_workers=config.num_workers,
                    sampler=train_sampler,
                    pin_memory=True,
                    collate_fn=collate_func_lambda)
#train_loader = torch.utils.data.DataLoader(
#                    train_dataset,
#                    batch_size=config.batch_size,
#                    num_workers=config.num_workers,
#                    sampler=val_set,
#                    pin_memory=True,
#                    collate_fn=collate_func_lambda)
val_loader = torch.utils.data.DataLoader(
                    train_dataset,
                    batch_size=config.batch_size,
                    num_workers=config.num_workers,
                    sampler=val_set,
                    pin_memory=True,
                    collate_fn=collate_func_lambda)

test_loader = torch.utils.data.DataLoader(
                    train_dataset,
                    #batch_size=config.batch_size,
                    batch_size=config.batch_size,
                    num_workers=config.num_workers,
                    sampler=test_set,
                    pin_memory=True,
                    collate_fn=collate_func_lambda)


print( "config.device:", config.device )
print( "学習セット数:",len( train_loader ) )
print( "評価セット数:",len( val_loader ))
print( "テストセット数:",len( test_loader ))
print( "use_amp:", config.use_amp )
print( "use_saved_pth:", config.use_saved_pth )

# --- 4. 最適化の実行 ---
if __name__ == "__main__":
    # 報酬を最大化したいのでdirection="maximize"
    file_path = "./optuna_journal_storage_20260830.log"
    
    # Create the storage object using JournalFileBackend
    storage = JournalStorage(JournalFileBackend(file_path))
    #study = optuna.create_study(direction="maximize", storage="sqlite:///db.sqlite3", study_name="quadratic-simple6" )
    pruner = optuna.pruners.PercentilePruner(
        percentile= 65.0, 
        n_startup_trials=5
    )
    study = optuna.create_study(direction="maximize", study_name="study91", storage=storage, 
                                pruner = pruner, load_if_exists=True )
    #study = optuna.create_study(direction="maximize", study_name="example-study15", storage=storage, 
    #                            pruner = optuna.pruners.MedianPruner() )
    study.optimize(objective, n_trials=41, n_jobs = 1 ) # 残り41回試行

    print("Best params:", study.best_params)
    print("Best reward:", study.best_value)

